# 03 — Climate & Spatial Research

**Stage 7 (Spatial features)** of `docs/PROJECT_BLUEPRINT.md` Phase 3. Stage 8
(climate feature engineering, Tracks A–D) shares this notebook and is added in a
follow-up pass — this build covers Stage 7 in full.

**What this notebook establishes:**
1. How many spatial clusters to use, and why (silhouette/inertia *and* the practical
   row-count-balance consequence for Tier 2 CV — not silhouette alone).
2. What actually separates the clusters (climate profile, dominant zone, size).
3. That the resulting `spatial_cluster` column genuinely retires the placeholder
   k=8 coordinate-only clustering `src/climate_health/evaluation/cv.py`'s
   `tier2_splits` has been falling back to since Stage 5.
4. A quick look at the raw coordinate polynomial features and the location-string
   fallback, both included in `src/climate_health/features/spatial.py` alongside the
   cluster assignment.

**Why this stage matters more than most:** Stage 3's forensics found zero exact
coordinate overlap between Train (43 unique coordinates) and Test (12), and Stage 4's
adversarial validation found `latitude`/`longitude`/`elevation`/`slope` alone separate
Train from Test almost perfectly (AUC 0.95–1.00). Every test coordinate the model will
ever see is geographically novel. This notebook is about building the one feature
family specifically designed to generalize to that novelty — not memorize the 43
training places.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from climate_health.data.loaders import load_train_full, load_test_full
from climate_health.evaluation.cv import (
    compute_twin_groups,
    merge_groups_to_respect_constraint,
    tier1_splits,
    tier2_splits,
)
from climate_health.features.spatial import (
    SpatialClusterFeaturizer,
    add_coordinate_polynomial_features,
    build_location_profiles,
    location_fallback_tokens,
    select_n_clusters,
    summarize_clusters,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42

train_full = load_train_full()
test_full = load_test_full()
print(f"train_full: {train_full.shape}, test_full: {test_full.shape}")
print(f"unique train coordinates: {build_location_profiles(train_full).shape[0]}")
print(f"unique test coordinates:  {build_location_profiles(test_full).shape[0]}")

## 1. Cluster count selection

`select_n_clusters` fits KMeans on standardized, per-location profiles
(`latitude`, `longitude`, `tavg_30d`, `rain_sum_90d`, `elevation` — averaged per unique
coordinate, so a location with many mortality records doesn't get extra clustering
weight just for being over-represented) across a range of `k`, and scores each by
silhouette and inertia.

In [ ]:
best_k_wide, results_wide = select_n_clusters(train_full, k_range=range(5, 13))
results_df = pd.DataFrame([r.__dict__ for r in results_wide])

# blueprint's stated range is 5-8; derive its silhouette-best k from the same results
# rather than a second, redundant select_n_clusters call.
narrow_results_df = results_df[results_df["k"] <= 8]
best_k_narrow = int(narrow_results_df.loc[narrow_results_df["silhouette"].idxmax(), "k"])

results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(results_df["k"], results_df["silhouette"], marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("silhouette score"); axes[0].set_title("Silhouette vs k")
axes[1].plot(results_df["k"], results_df["inertia"], marker="o")
axes[1].set_xlabel("k"); axes[1].set_ylabel("inertia"); axes[1].set_title("Inertia (elbow) vs k")
plt.tight_layout()
plt.show()
print(f"Silhouette-best k in blueprint's stated 5-8 range: {best_k_narrow}")

**Reading this so far, before looking past silhouette alone:** silhouette peaks at
k=5 and declines roughly monotonically through k=12 (a small bump at k=11 isn't
meaningful with only 43 points feeding the metric). Taken alone, this would suggest
k=5. But silhouette only measures how cleanly separated the clusters are in feature
space — it says nothing about whether the resulting clusters are *usable* for Tier 2
CV, which is exactly the check `docs/PROJECT_BLUEPRINT.md` Stage 7 calls for before
trusting a cluster-count choice. That check is next.

## 2. The row-count-balance consequence (why silhouette alone is misleading here)

Stage 5's `tier2_splits` groups by spatial cluster (merged with the twin-record
constraint) and needs a reasonably balanced set of groups to produce usable folds — the
review that fixed `tier2_splits`'s fold-assignment bug also found that a single
dominant group structurally caps how balanced *any* fold assignment can be, no matter
how good the bin-packing algorithm is. So cluster count needs to be judged by the
resulting group-size distribution, not silhouette in isolation.

In [ ]:
twin_groups = compute_twin_groups(train_full, keys=("location", "deathdate"))

balance_rows = []
for k in range(5, 9):
    feat = SpatialClusterFeaturizer(n_clusters=k, random_state=RANDOM_STATE)
    labeled = feat.fit_transform(train_full)
    final_groups = merge_groups_to_respect_constraint(labeled["spatial_cluster"].to_numpy(), twin_groups)
    sizes = pd.Series(final_groups).value_counts()
    balance_rows.append(
        {
            "k": k,
            "silhouette": next(r.silhouette for r in results_wide if r.k == k),
            "n_groups_after_twin_merge": len(sizes),
            "largest_group_frac": sizes.max() / len(train_full),
            "smallest_group_frac": sizes.min() / len(train_full),
            "group_sizes": sorted(sizes.tolist(), reverse=True),
        }
    )

balance_df = pd.DataFrame(balance_rows)
balance_df

**This changes the recommendation.** At k=5 (silhouette's top pick), one cluster
alone holds ~84% of all training rows — Tier 2 CV at k=5 would barely test geographic
generalization at all, since four of five "geographic" groups are tiny and the model
would mostly just be validated against held-in/held-out slices of one giant blob. From
k=6 onward the largest group's share drops sharply (to roughly half the data) and stays
in a similar range through k=8 — silhouette keeps declining across that range, but only
mildly, while the practical Tier 2 usability improves a lot.

**Decision: use k=6.** It's the smallest k where the dominant-cluster problem is
substantially resolved (largest group ≈52% vs. 84% at k=5), it stays within the
blueprint's stated 5–8 range, its silhouette (second-best among 5–8) is still
respectable, and a coarser clustering is preferable to a finer one on general
principle when both are defensible (Occam's razor, and fewer, larger clusters are
easier to interpret and describe in the model card later). This is exactly the kind of
call Stage 7 asks to be made on Tier 2 behavior, not silhouette alone — recorded here so
the reasoning is auditable, not just the number.

In [ ]:
N_CLUSTERS = 6
spatial_featurizer = SpatialClusterFeaturizer(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE)
train_labeled = spatial_featurizer.fit_transform(train_full)
test_labeled = spatial_featurizer.transform(test_full)

print("Train cluster sizes:")
print(train_labeled["spatial_cluster"].value_counts().sort_index())
print("\nTest cluster sizes:")
print(test_labeled["spatial_cluster"].value_counts().sort_index())

## 3. Framework-validity sanity check across k

Before committing, a quick check that Tier 1 vs. Tier 2 still behaves the way it
should at k=6 — the same synthetic-target framework-correctness idea from
`tests/test_cv.py`'s capstone test: build a target that's *purely* a function of
spatial cluster, and confirm a coordinate-based classifier scores near-perfectly under
Tier 1 (no geography guard) but collapses toward chance under Tier 2 (which holds
whole clusters out). If that gap doesn't appear at k=6, the clustering isn't
structured enough for Tier 2 to do its job.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = train_labeled.copy()
cluster_sizes = df["spatial_cluster"].value_counts().sort_values(ascending=False)
group_totals = {0: 0, 1: 0}
cluster_labels = {}
for cluster_id, size in cluster_sizes.items():
    smaller_group = 0 if group_totals[0] <= group_totals[1] else 1
    cluster_labels[cluster_id] = smaller_group
    group_totals[smaller_group] += size
df["_synthetic_target"] = df["spatial_cluster"].map(cluster_labels)
print("synthetic target global rate:", df["_synthetic_target"].mean())

def _score(train_idx, val_idx, X, y):
    clf = KNeighborsClassifier(n_neighbors=1)
    clf.fit(X[train_idx], y[train_idx])
    return accuracy_score(y[val_idx], clf.predict(X[val_idx]))

X = df[["latitude", "longitude"]].to_numpy()
y = df["_synthetic_target"].to_numpy()

tier1_scores = [
    _score(tr, va, X, y)
    for tr, va in tier1_splits(df, target_col="_synthetic_target", n_splits=5, random_state=RANDOM_STATE)
]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tier2_scores = [
        _score(split.train_idx, split.val_idx, X, y)
        for split in tier2_splits(
            df, spatial_cluster_col="spatial_cluster", n_splits=5, n_repeats=3, random_state=RANDOM_STATE
        )
    ]

print(f"Tier 1 (random) accuracy:     mean={np.mean(tier1_scores):.3f}  (n={len(tier1_scores)})")
print(f"Tier 2 (geographic) accuracy: mean={np.mean(tier2_scores):.3f} std={np.std(tier2_scores, ddof=1):.3f}  (n={len(tier2_scores)})")

**Interpreted after running (actual result: Tier 1 mean=1.000, Tier 2
mean≈0.27, vs. a synthetic-target base rate of ≈0.48):** Tier 1 hits perfect
accuracy, as expected — under a twin-guarded-but-not-geography-guarded split, a
1-nearest-neighbor classifier almost always has a training point at (or extremely
close to) the exact same coordinate as each validation point, so it trivially
"predicts" the cluster-derived label correctly every time.

The more interesting result is Tier 2: accuracy lands *below* the ≈0.48 base rate,
not just near it. This is actually a **sharper** demonstration of the framework
working than a plain collapse-to-chance would have been. Here's why: the synthetic
target was assigned per-cluster by size-balancing, not by geographic adjacency, so a
held-out cluster's nearest surviving neighbor cluster is roughly a coin flip on
whether it happens to share the same synthetic label. When 1-NN is forced to
extrapolate from neighboring clusters for an entire held-out cluster at once, it
isn't "unsure" the way random guessing would be — it's *confidently and
systematically wrong* whenever the nearest surviving cluster carries the opposite
label, dragging the average below chance. This is exactly the failure mode Tier 2
exists to catch: a model that has learned to key off *which specific training points
are nearby* rather than anything that transfers to new geography fails hard, not
just moderately, on genuinely unseen locations. **The gap (1.000 vs. ≈0.27) confirms
the k=6 clustering is structured enough for Tier 2 to meaningfully separate
"generalizes" from "memorizes," now validated against Stage 7's real spatial feature
rather than the old coordinate-only placeholder.**

## 4. What separates the clusters

`summarize_clusters` gives the per-cluster profile: record count, unique-location
count, dominant zone, and the mean of every feature that went into the clustering.

In [ ]:
summarize_clusters(spatial_featurizer, train_full)

In [ ]:
centers = spatial_featurizer.cluster_centers_original_scale()
centers.index.name = "spatial_cluster"
centers

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(
    train_labeled["longitude"], train_labeled["latitude"],
    c=train_labeled["spatial_cluster"], cmap="tab10", s=40, alpha=0.7, label="train"
)
ax.scatter(
    test_labeled["longitude"], test_labeled["latitude"],
    c=test_labeled["spatial_cluster"], cmap="tab10", s=90, marker="^",
    edgecolor="black", linewidth=0.8, label="test (assigned by nearest centroid)"
)
ax.scatter(centers["longitude"], centers["latitude"], c="black", marker="x", s=120, label="centroids")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Spatial clusters (k={N_CLUSTERS}) — train (circles) vs test (triangles)")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

**Interpreted after running:** the plot is the most direct evidence of what this
whole stage is for — every test triangle sits at a coordinate that was never part of
`fit()`, and each is nonetheless placed near whichever training cluster's coordinates
*and* climate profile it most resembles, not left unclassified or forced into an
arbitrary nearest-training-point lookup. Compare this to what `location_fallback_tokens`
would do with a district string that has literally no match in training — nothing
meaningful, since string equality has no notion of "closest."

## 5. Raw coordinate polynomial features — a quick risk check

`add_coordinate_polynomial_features` adds `latitude²`, `longitude²`, `latitude×longitude`
per the blueprint. Two different questions matter here, and it's worth being precise
about which one is being asked, since Stage 4 already answered a related-but-different
one:

- Stage 4's adversarial validation asked "can a model tell a Train row from a Test row
  using this feature" — the answer there was yes, almost perfectly, because Train's 43
  coordinates and Test's 12 simply occupy different numeric ranges (0% exact overlap).
  That's a statement about *dataset membership*, not about the actual competition
  target.
- The check below asks a different question: "does this coordinate feature, on its
  own, predict `is_climate_sensitive`?" — checked two ways, since they can disagree.
  A **linear** model (logistic regression on one raw coordinate) can only pick up a
  monotonic relationship; a **1-nearest-neighbor** model can effectively memorize which
  exact training point a value is closest to, which is a much closer proxy for what a
  flexible tree-based model *could* do if allowed to overfit to location. Comparing the
  two exposes memorization risk that the linear check alone would hide.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

poly = add_coordinate_polynomial_features(train_full)
target = poly["is_climate_sensitive"]

rows = []
single_cols = ["latitude", "longitude", "latitude_sq", "longitude_sq", "latitude_x_longitude"]
for col in single_cols:
    X_col = poly[[col]]
    linear_auc = roc_auc_score(target, LogisticRegression(max_iter=1000).fit(X_col, target).predict_proba(X_col)[:, 1])
    knn = KNeighborsClassifier(n_neighbors=1).fit(X_col, target)
    knn_auc = roc_auc_score(target, knn.predict_proba(X_col)[:, 1])
    rows.append({"feature": col, "linear_AUC_in_sample": linear_auc, "1nn_AUC_in_sample": knn_auc})

# also check the *joint* (latitude, longitude) pair -- i.e. exact coordinate identity,
# the strongest possible "does knowing the precise location alone predict the target"
# check, since it's what a model would need to actually memorize a location lookup.
X_joint = poly[["latitude", "longitude"]]
joint_knn = KNeighborsClassifier(n_neighbors=1).fit(X_joint, target)
joint_auc = roc_auc_score(target, joint_knn.predict_proba(X_joint)[:, 1])
rows.append({"feature": "(latitude, longitude) joint", "linear_AUC_in_sample": float("nan"), "1nn_AUC_in_sample": joint_auc})

pd.DataFrame(rows).sort_values("1nn_AUC_in_sample", ascending=False)

**Interpreted after running (actual result: every linear AUC ≈0.50–0.51, every
1-NN AUC — including the joint (latitude, longitude) pair — also ≈0.50–0.52, not
meaningfully higher):** this is a more informative result than either half of the
original hypothesis (which was: linear should be weak, 1-NN should be strong from
memorization). Neither is strong, and the reason is worth stating plainly, because
it's a genuinely useful finding, not a non-result.

A 1-nearest-neighbor classifier evaluated on its own training data would hit a
memorization ceiling *if* each coordinate mapped to one consistent outcome — but it
doesn't here, even using the exact `(latitude, longitude)` pair. With only 43 unique
training coordinates spread across 3,146 records, each location averages ~73 records,
and — the key point — those ~73 records at a given location do **not** all share the
same target value. Because 1-NN with tied (identical) coordinate distances effectively
draws from that location's own mixed record history rather than "looking up itself,"
its in-sample AUC stays near chance too.

This is exactly consistent with Stage 3's forensics headline finding (no single
feature or pair comes anywhere near a deterministic 0.95+ AUC; the best pair found was
0.764), now confirmed from the geography side specifically: **exact location identity
alone is not a target lookup table.** That's reassuring for the raw coordinate and
polynomial features going into Phase 3's feature set — the risk Stage 4's adversarial
validation flagged was about *distinguishing train from test rows* (a genuinely
different, real risk for generalization, still worth watching under Tier 2), not about
location being able to leak the actual answer. Per §0.2's ruling, these features are
kept, watched under Tier 2 in Phase 4, and this check is the documented reason they
weren't excluded here on a guess.

## 6. Location-string fallback — confirming it's genuinely secondary

A last check on `location_fallback_tokens`: how much of test's location-string
structure is actually novel relative to train, which is the concrete reason this stays
a documented fallback rather than the primary geographic feature.

In [ ]:
train_tokens = location_fallback_tokens(train_full, max_tokens=3)
test_tokens = location_fallback_tokens(test_full, max_tokens=3)

for i in range(3):
    col = f"location_token_{i}"
    train_vals = set(train_tokens[col].dropna())
    test_vals = set(test_tokens[col].dropna())
    novel = test_vals - train_vals
    print(f"{col}: {len(test_vals)} unique test values, {len(novel)} never seen in train ({novel if novel else '—'})")

**Interpreted after running:** whatever fraction of test's district/region tokens
are entirely novel is exactly the fraction of test rows this fallback feature could not
meaningfully place, versus `SpatialClusterFeaturizer`'s 100% coverage of every test
coordinate via nearest-centroid assignment. This is the concrete version of the
abstract argument in the module docstring, not just an assertion.

## 7. Summary & decision record

- **Cluster count: k=6.** Silhouette alone favors k=5, but k=5 produces a single
  dominant cluster holding ~84% of training rows, which would make Tier 2 CV
  nearly meaningless (one geographic group barely held out at all). k=6 is the
  smallest k in the blueprint's 5–8 range where that imbalance is substantially
  resolved (largest group ≈52%), while still scoring reasonably on silhouette.
- **Clustering basis: coordinates + climate normals jointly** (`tavg_30d`,
  `rain_sum_90d`, `elevation`), not coordinates alone — this is what lets a genuinely
  novel test coordinate be placed by climate similarity rather than treated as an
  unplaceable outlier or forced onto the single nearest training point regardless of
  how climatically different it is.
- **Stability: per-coordinate, not per-record.** Every row sharing a coordinate gets
  the identical cluster label, so `spatial_cluster` behaves as a location-level
  identity — consistent with how `zone` is already treated in `cv.py`'s Tier 3 holdout
  logic, and what makes a `spatial_cluster × zone` interaction (Stage 9) a coherent
  thing to build.
- **The placeholder is retired.** `tier2_splits` no longer needs to fall back to
  `preliminary_kmeans_k8` — every subsequent Tier 2 CV run should pass
  `spatial_cluster_col="spatial_cluster"` (from this featurizer, fit on train) and
  `spatial_cluster_source="spatial_py_kmeans_k6_coord_plus_climate"` explicitly, so any
  historical CV numbers logged against the placeholder are never silently compared
  against these.
- **Raw lat/lon and polynomial coordinate features are kept**, not dropped, despite
  carrying strong train/test-distinguishing signal — per §0.2's ruling, that signal is
  a reason to watch these features under Tier 2, not to exclude them before a model
  ever gets to use them.
- **The location-string fallback stays secondary.** It exists in `spatial.py` for
  completeness and as a documented brittle option, but the cluster assignment is the
  feature everything downstream should actually use.

**Next:** Stage 8 (climate feature engineering, Tracks A–D) extends this notebook,
building anomaly and ratio features on top of the cluster-level climate normals this
stage established.

## 8. Executive Summary

**Scope.** This notebook builds and validates Stage 7's spatial feature representation
(`src/climate_health/features/spatial.py`), the direct response to the single most
important structural fact in this dataset: Train's 43 unique coordinates and Test's 12
have **zero exact overlap** (Stage 3 forensics), and Stage 4's adversarial validation
found `latitude`/`longitude`/`elevation`/`slope` alone separate Train from Test almost
perfectly (AUC 0.95–1.00). Every coordinate the model will be scored on at competition
close is geographically novel relative to training. Everything below either builds the
feature designed to generalize to that novelty, or checks that it actually does.

### Key findings

**1. Cluster count: silhouette alone gives the wrong practical answer.** Silhouette
score peaks at k=5 (0.436) and declines through k=12 — on that metric alone, k=5 wins.
But k=5 places **84.3%** of all training rows in a single cluster (2,653 of 3,146),
leaving four other "geographic" groups holding as few as 54–158 rows each. A Tier 2 CV
split under that clustering would barely test geographic generalization at all, since
one blob dominates every fold. Checking the row-count-balance consequence directly
(the same structural-bound logic used to diagnose and fix `cv.py`'s Tier 2 fold-size
bug in the prior review) showed k=6 is the smallest step in the blueprint's stated 5–8
range where this resolves substantially: the largest cluster drops to 51.8% (1,631
rows) and the size distribution is materially more even (`[1631, 1172, 158, 128, 54,
3]` vs. k=5's `[2653, 158, 153, 128, 54]`). **Decision: k=6**, silhouette-informed but
not silhouette-dictated — exactly the kind of judgment call docs/PROJECT_BLUEPRINT.md
Stage 7 asks to be made on Tier 2 usability, not an unsupervised metric in isolation.

**2. The framework-validity check confirms Tier 2 can actually detect geographic
memorization at k=6 — and more sharply than expected.** A 1-nearest-neighbor
classifier trained to recover a purely cluster-derived synthetic target hit perfect
accuracy (1.000) under Tier 1 (twin-guarded, not geography-guarded) but fell to 0.265
under Tier 2 — not just toward the ≈0.48 chance level, but *below* it. That's a sharper
result than a plain chance-collapse would have been: because the synthetic labels were
assigned by size-balancing rather than geographic adjacency, a model forced to
extrapolate from neighboring clusters to an entirely held-out one is often
*confidently wrong*, not merely unsure. This is precisely the failure mode Tier 2
exists to catch, now demonstrated against the real Stage 7 clustering rather than the
retired coordinate-only placeholder.

**3. Cluster 3 is a near-degenerate group worth flagging, not hiding.** At k=6, cluster
3 holds only 3 records from a single location. It survived the k selection because it
doesn't change the balance argument for k=6 vs. k=5 materially, but it's the one
cluster where Tier 2 CV will get essentially no independent signal — any fold
containing it as a held-out group is really testing "does the model handle 3 records
from one place reasonably," not geographic generalization in a statistical sense. This
is disclosed here so it isn't mistaken for a modeling artifact later if Phase 4 shows
unusually noisy Tier 2 folds.

**4. Test coordinates concentrate into just 3 of the 6 clusters (0, 4, 5) — none of the
single-location clusters (1, 2, 3).** This is worth watching, not alarming: it means
the 12 test coordinates' climate profiles happen to resemble the larger, more populous
training clusters more than the small outlier ones (e.g. cluster 1's high-elevation,
low-temperature profile at 1,660m). It's consistent with — not contradictory to — the
zero-coordinate-overlap finding; nearest-centroid assignment by climate similarity is
exactly what let every test row land somewhere sensible despite that.

**5. Cluster separation is real and climatically coherent, not an artifact.** The
`cluster_centers_original_scale()` table and the geographic scatter plot together show
clusters that differ meaningfully in elevation (1,127m to 1,660m), temperature
(20.2°C–23.5°C), and rainfall (263mm–478mm over 90 days) — not just in raw coordinates.
Every test triangle in the scatter plot lands near a training cluster with a similar
climate profile, which is the concrete, visual version of what "generalizes by climate
similarity, not lookup" means.

**6. Raw coordinates and their polynomial transforms carry no meaningful signal for
the actual target, linearly or via 1-NN memorization — a reassuring, verified finding,
not an assumption.** Every linear single-feature AUC sits at ≈0.50–0.51; every 1-NN AUC
(including the joint `(latitude, longitude)` pair) sits at ≈0.50–0.52, essentially
indistinguishable from chance. The reason is itself informative: with only 43 unique
training coordinates averaging ~73 records each, and those records *not* sharing a
consistent target outcome, exact location identity is not a target lookup table — a
finding fully consistent with Stage 3's forensics headline (no feature or pair
approaches deterministic prediction; best pair found was 0.764 AUC). This is a
materially different, non-contradictory finding from Stage 4's adversarial-validation
result: coordinates strongly indicate *dataset membership* (Train vs. Test) but do not
leak the *actual answer*. Both raw coordinates and their polynomial transforms are kept
in the feature set on this basis, per §0.2's ruling that a high adversarial-validation
signal is a reason to watch a feature under Tier 2, not to reflexively drop it before a
model gets to use it.

**Reproducibility note, disclosed for rigor:** the 1-NN AUC values in Finding 6 varied
slightly between the sandbox run used to write this notebook and the user's own
execution (e.g. joint-coordinate 1-NN AUC: 0.513 vs. 0.505) despite an identical
`random_state`. This is expected, not a bug: `KNeighborsClassifier` has no
`random_state`-controlled behavior of its own, and with as many exact-distance ties as
this data produces (43 unique coordinates shared across 3,146 rows), which tied
neighbor gets returned can depend on the underlying nearest-neighbor library's
internal tie-breaking, which is not guaranteed identical across platforms/BLAS
builds. Every observed value stayed within the same ≈0.50–0.52 chance band, so the
substantive conclusion is unaffected — but it's worth knowing this specific number is
not bit-for-bit reproducible before anyone is tempted to treat a small shift in it as
meaningful in a future re-run.

**7. The location-string fallback is confirmed genuinely inferior, not just
theoretically brittle.** Of test's 11 unique finest-grained location tokens, 10 were
never seen in training at all; of 6 unique district-level tokens, 3 were novel. A
representation that can only describe training strings would have no meaningful
placement for the large majority of test rows at the fine-grained level. `
SpatialClusterFeaturizer` covers 100% of test rows via nearest-centroid assignment by
construction — the concrete evidence behind the module's design choice, not merely its
stated rationale.

### Decisions recorded

- **k=6**, clustering jointly on `(latitude, longitude, tavg_30d, rain_sum_90d,
  elevation)`, decided on the row-count-balance and Tier 2 usability consequence, not
  silhouette score alone.
- **Cluster assignment is a stable per-coordinate attribute**, computed independently
  at fit time (train) and transform time (test/any other data) from each dataset's own
  climate profile — never borrowed across datasets, and never varying within a
  location's own record history.
- **The `cv.py` placeholder (`preliminary_kmeans_k8`) is retired.** Every future Tier 2
  CV run should pass `spatial_cluster_col="spatial_cluster"` (this featurizer's output,
  fit on train and applied consistently) and an explicit
  `spatial_cluster_source="spatial_py_kmeans_k6_coord_plus_climate"`, so no historical
  CV number logged against the old placeholder is ever silently compared against a
  post-Stage-7 one.
- **Raw lat/lon and their polynomial transforms are kept in the feature set**, watched
  under Tier 2 in Phase 4 rather than excluded now, on the basis of the verified (not
  assumed) finding in \#6 above.
- **The location-string fallback (`location_fallback_tokens`) stays secondary/optional**
  — available for interpretability or as a coarse categorical if Phase 4 ever wants it,
  but not part of the primary spatial representation.

### Implications for the rest of the project

- **Stage 8 (climate feature engineering, next)** can build directly on this stage's
  per-coordinate climate-normal machinery (`build_location_profiles`): anomaly features
  ("how far is this record's `tavg_30d` from its *cluster's* normal") now have a
  well-defined, non-degenerate baseline to compute against, more stable than trying to
  use each of the 43 raw locations individually (several of which have very few
  records).
- **Stage 9 (interactions & encoding)** can treat `spatial_cluster` as a legitimate
  location-level categorical for a `spatial_cluster × zone` interaction and for
  out-of-fold target encoding, on the same footing as `zone` itself — a direct
  consequence of the per-coordinate stability design choice in Finding 5/Decision 2.
- **Phase 4 (baselines & model zoo)** inherits a `spatial_cluster` column that
  materially changes what a "trustworthy" Tier 2 number looks like going forward: any
  Tier 2 CV result computed before this stage (against the placeholder) is not
  comparable to any computed after it, and that boundary should be marked explicitly
  in `docs/experiment_registry.md` once Stage 16 exists.
- **A concrete open risk carried forward, not resolved here:** cluster 3's near-singleton
  status (3 records) means Tier 2's `min`/`std` reporting (per Stage 5's design) will
  likely be dominated by whichever fold contains it. This isn't a defect in the
  clustering — it reflects genuine geographic sparsity in the raw data — but it's worth
  watching in Phase 4's model comparison table rather than being surprised by an
  unusually wide Tier 2 spread later.
- **No action required on raw coordinates/polynomial features before Phase 4** — they
  are verified low-risk for direct target leakage (Finding 6) and their fate as
  production features is deferred to real Tier 2 evidence once actual models are
  trained, not decided here on a proxy check.

### Overall verdict

Stage 7 is **complete and validated**: the spatial representation generalizes to novel
coordinates by construction and by demonstration (scatter plot, test cluster coverage),
the cluster count decision is evidence-based and documented rather than defaulted, the
CV framework is confirmed to actually detect geographic memorization against this real
clustering, and every claim in this summary traces to a number produced by code in this
notebook — not an assumption carried in from an earlier stage. Ready to proceed to
Stage 8.

## 9. Climate feature engineering (Stage 8)

Stage 7 established `train_labeled`/`test_labeled` (spatial cluster assigned) and
the per-coordinate climate-normal machinery those clusters were fit on. Stage 8
builds directly on top: four tracks (A–D) plus the leakage-timing provenance table,
per `docs/PROJECT_BLUEPRINT.md`.

**Track A — provided data, made right.** Drop `hot_days_30d` (re-verified constant
here, not just assumed from Stage 3) and add normalized/per-day + ratio rainfall
features.

In [ ]:
from climate_health.features.climate import (
    ClimateAnomalyFeaturizer,
    add_heat_exceedance_features,
    add_ndvi_trend_feature,
    add_rainfall_rate_features,
    compute_heat_threshold,
    drop_dead_columns,
    render_provenance_markdown,
)

train_a = drop_dead_columns(train_labeled)
test_a = drop_dead_columns(test_labeled)
train_a = add_rainfall_rate_features(train_a)
test_a = add_rainfall_rate_features(test_a)

rain_new_cols = [
    "rain_rate_7d",
    "rain_rate_30d",
    "rain_rate_90d",
    "rain_ratio_acute_medium",
    "rain_ratio_medium_chronic",
    "rain_intensity_30d",
    "rain_day_fraction_30d",
]
print("Track A — rainfall rate/ratio features, train summary:")
print(train_a[rain_new_cols].describe().T[["mean", "std", "min", "max"]])
print("\nNaN counts (genuinely undefined ratios, zero-denominator rows):")
print(train_a[rain_new_cols].isna().sum())

**Reading this:** the NaN counts land exactly on `rain_ratio_acute_medium` and
`rain_intensity_30d` — both have `rain_sum_30d` in the denominator, which is exactly
zero for a handful of real training rows. `_safe_ratio` returns NaN there rather than
inf or a fabricated 0, which is the honest answer ("undefined", not "no rain
concentration") and is handled natively by every tree-based model in this project's
zoo. `rain_ratio_medium_chronic` has no NaNs because `rain_sum_90d` is never zero in
the observed data.

**Track B — temperature: a data-driven heat-exceedance threshold.** The traditional
35°C "extreme heat" convention would silently reproduce `hot_days_30d`'s exact
dead-column problem here — `tmax_30d` never actually reaches 35°C in this dataset.
The threshold below is fit on **train only** (never test, never combined) as a
percentile of `tmax_30d`, then reused verbatim for both.

In [ ]:
heat_threshold = compute_heat_threshold(train_a, quantile=0.90)
print(f"Heat threshold (90th percentile of train tmax_30d): {heat_threshold:.2f}°C")
print(f"Train tmax_30d max: {train_a['tmax_30d'].max():.2f}°C  "
      f"(traditional 35°C convention would never fire)")

train_b = add_heat_exceedance_features(train_a, threshold=heat_threshold)
test_b = add_heat_exceedance_features(test_a, threshold=heat_threshold)

print(f"\nTrain exceedance flag rate: {train_b['tmax_30d_exceeds_flag'].mean():.3%}")
print(f"Test exceedance flag rate:  {test_b['tmax_30d_exceeds_flag'].mean():.3%}")
print("\nExceedance margin summary (train):")
print(train_b["tmax_30d_exceedance_margin"].describe())

**Reading this:** ~9.9% of training windows and ~11.9% of test windows exceed the
learned threshold — similar enough between train/test to suggest the threshold
generalizes reasonably (it isn't overfit to a train-only quirk), while still being
selective enough to be informative rather than firing on nearly everything or
nothing. The continuous `_exceedance_margin` feature carries information on both
sides of the cutoff, not only at the single binary boundary the flag alone gives.

**Track B/C shared — learned climate anomalies.** `ClimateAnomalyFeaturizer` computes
`{col}_anomaly` = value minus a *train-only-learned* baseline, with a 3-tier fallback
per row: `(spatial_cluster, month)` mean → `spatial_cluster`-only mean → global
training mean. Applied here to one temperature column (Track B) and one rainfall
column (Track C) as a single shared pass.

In [ ]:
anomaly_featurizer = ClimateAnomalyFeaturizer(
    value_cols=["tavg_30d", "rain_sum_90d"],
    group_col="spatial_cluster",
    min_group_month_size=5,
)
train_c = anomaly_featurizer.fit_transform(train_b)
test_c = anomaly_featurizer.transform(test_b)

print(f"(cluster, month) buckets meeting min_group_month_size=5: "
      f"{anomaly_featurizer.n_group_month_buckets_fit_} / "
      f"{anomaly_featurizer.n_group_month_buckets_total_} non-empty buckets seen in training")

print("\nBaseline tier used, train rows:")
print(train_c["tavg_30d_anomaly_baseline_level"].value_counts())
print("\nBaseline tier used, test rows:")
print(test_c["tavg_30d_anomaly_baseline_level"].value_counts())

print("\ntavg_30d_anomaly summary (train):")
print(train_c["tavg_30d_anomaly"].describe())
print("\nrain_sum_90d_anomaly summary (train):")
print(train_c["rain_sum_90d_anomaly"].describe())

**Reading this:** the sparse-bucket problem is real, not hypothetical — a
meaningful share of non-empty `(cluster, month)` buckets fall below the
`min_group_month_size=5` trust threshold, so a real fraction of rows correctly fall
back to the coarser `spatial_cluster`-only baseline rather than a seasonally-specific
one computed from too few records to be reliable. The `global` tier is available as a
third-level defensive fallback but is not expected to fire here in practice: because
`SpatialClusterFeaturizer` assigns every test coordinate to its nearest training
centroid by construction, `spatial_cluster` itself is never a genuinely novel category
at transform time — it only would be if this featurizer were ever applied to a
`spatial_cluster` value it never saw at all (verified directly in
`tests/test_climate.py`, not merely assumed).

**Track D (stretch) — NDVI short-term trend.**

**Adversarial review finding, fixed before shipping:** the anomaly baseline
computed by `fit_transform` (used to build the *training* set's own anomaly
features, exactly as run above) initially used each bucket's plain in-sample mean —
which lets a training row's own value contribute to its own baseline. For a large
group this barely matters, but for `spatial_cluster` 3 (only 3 training records),
this shrank the true anomaly toward zero by a full 33% (verified: in-sample anomalies
of `[0.296, 0.445, -0.741]` vs. the honest leave-one-out values `[0.444, 0.668,
-1.111]` for those same 3 rows — the in-sample version was 2/3 of the correct
magnitude, exactly the `(n-1)/n` shrinkage factor predicted for `n=3`). This does not
leak the *target* — it's a bias in a covariate summary statistic, the same category
as any `fit_transform` that reuses training statistics on training rows (e.g.
`StandardScaler`) — but it disproportionately weakens the anomaly signal for
precisely the small, sparse groups (cluster 3, and any `(cluster, month)` bucket at
the `min_group_month_size` threshold) where a real anomaly matters most.
`ClimateAnomalyFeaturizer.fit_transform` now computes true leave-one-out baselines
for the rows it is fit on — each row's own value is excluded from its own bucket
statistic, falling back a tier if removing it would leave the bucket empty — while
`transform()` on genuinely new data (test/production, as used below) is unchanged
and was never affected, since those rows never contributed to the baseline they look
up. `tests/test_climate.py` now directly verifies the leave-one-out arithmetic
(hand-computed against a synthetic 3-row group) and the tier-fallback edge case
where excluding a bucket's lone member would divide by zero.

**A second review finding, worth flagging rather than "fixing" — a genuine design
consideration, not a bug:** `spatial_cluster` (the `group_col` used above) was
itself built by Stage 7's `SpatialClusterFeaturizer` by clustering each *location's*
average `tavg_30d` and `rain_sum_90d` (among other columns) — the same two columns
this anomaly featurizer computes anomalies for. This means clusters were formed, in
part, specifically to group together locations with *similar* average temperature
and rainfall, so a `spatial_cluster`-relative anomaly will tend to be smaller than
if a coarser, climate-independent grouping (e.g. `zone`) were used instead — some of
the "anomaly" signal is structurally absorbed into the cluster assignment itself
rather than surviving as a residual. This isn't wrong — the feature still captures
genuine record-level (day-to-day, season-to-season) deviation that the location-level
clustering never saw — but it means the *magnitude* of this feature's lift should be
interpreted with that structural relationship in mind. **Recommended for Phase 4:**
compare `group_col="spatial_cluster"` against `group_col="zone"` (or `location`) as
an ablation once real Tier 2 CV numbers exist, rather than assuming the finer-grained
clustering is automatically the better anomaly baseline.

In [ ]:
train_d = add_ndvi_trend_feature(train_c)
test_d = add_ndvi_trend_feature(test_c)

print("ndvi_trend_30_90 summary (train):")
print(train_d["ndvi_trend_30_90"].describe())
print(f"\nGreening rows (positive trend): {(train_d['ndvi_trend_30_90'] > 0).mean():.1%}")
print(f"Browning rows (negative trend): {(train_d['ndvi_trend_30_90'] < 0).mean():.1%}")

train_climate_final = train_d
test_climate_final = test_d
print(f"\nFinal shapes — train: {train_climate_final.shape}, test: {test_climate_final.shape}")

**Reading this:** only two NDVI windows are provided (30d, 90d composites), so
this is a simple difference, not a fitted slope over a real time series — scoped to
what the data actually supports.

## 10. Provenance / leakage-timing table

Every provided and derived climate feature, with its reference period, prediction-time
availability, and risk level — the explicit Stage 8 deliverable per the blueprint,
framed as the eventual enforced boundary for Stage 19's deployment demo.

In [ ]:
from IPython.display import Markdown, display

provenance_md = render_provenance_markdown()
display(Markdown(provenance_md))

In [ ]:
with open("../docs/feature_provenance.md", "w") as f:
    f.write("# Feature Provenance & Leakage-Timing Table\n\n")
    f.write(
        "Generated by `climate_health.features.climate.render_provenance_markdown()` "
        "(Stage 8). Covers every provided and derived climate feature: reference "
        "period, availability at prediction time, leakage risk, and notes.\n\n"
    )
    f.write(provenance_md)
print("Saved docs/feature_provenance.md")

## 11. Summary & decision record

- **Track A (rainfall rate/ratio):** `hot_days_30d` re-verified constant and dropped;
  7 new rate/ratio features added, all using `_safe_ratio` so a zero-denominator row
  produces an honest NaN rather than an inf or a fabricated 0.
- **Track B (heat exceedance): threshold is data-driven, not the 35°C convention.**
  Verified `tmax_30d` never reaches 35°C in this dataset, so a fixed-degree threshold
  would have reproduced `hot_days_30d`'s exact dead-column problem under a new name.
  The 90th-percentile train-only threshold (~30.7°C) gives a train/test flag-rate gap
  small enough (9.9% vs. 11.9%) to suggest it generalizes.
- **Track B/C (anomalies): 3-tier train-only-learned baseline**, mirroring Stage 7's
  leakage-safety pattern exactly (fit-on-train, applied to any dataset via lookup —
  never recomputed independently per-dataset). The `(cluster, month)` sparsity that
  motivated the group-level fallback tier is empirically confirmed here, not assumed.
- **Track D (NDVI trend):** a straightforward two-point difference, deliberately not
  oversold as a fitted trend the data can't support.
- **Two features were deliberately *not* built**, and are documented as limitations
  rather than shipped as something that looks like it measures what it doesn't:
  a true day-count reconstruction of `hot_days_30d` (no daily series available, only
  30-day window scalars), and a consecutive dry-spell length feature (`rain_days_30d`
  is a count, not the positions of rainy days).
- **Adversarial review (post-build) found and fixed a real self-inclusion bias** in `ClimateAnomalyFeaturizer.fit_transform`: bucket means originally included each row's own value, shrinking small-group anomalies toward zero (verified: 33% shrinkage for cluster 3's 3 training rows). `fit_transform` now computes true leave-one-out baselines for the rows it is fit on; `transform()` on external data was already correct and is unchanged. The review also flagged a non-bug design consideration: `spatial_cluster` was itself built partly from each location's average `tavg_30d`/`rain_sum_90d`, so cluster-relative anomalies structurally absorb some of the signal the clustering was optimized to minimize — recommended as a Phase 4 ablation (`group_col="zone"` vs. `"spatial_cluster"`), not a defect to fix here. A bare-string `value_cols` now raises a clear `TypeError` instead of silently decomposing into characters, and `add_heat_exceedance_features` now rejects a non-finite `threshold`.
- **Provenance table saved to `docs/feature_provenance.md`** — the leakage-timing
  reference that Stage 9 (interactions/encoding) and Phase 4+ modeling should consult
  before treating any derived feature as safe by default.

**Next:** Stage 6 (temporal/demographic features) and Stage 9 (interactions &
encoding) can now both build on `spatial_cluster` and this stage's climate-anomaly
machinery — `spatial_cluster × zone` interactions and out-of-fold target encoding are
the natural next step once Stage 6 lands the temporal side.

## 12. Executive Summary — Stage 8 addendum

**Scope.** This addendum covers Stage 8 (climate feature engineering & enrichment),
built directly on Stage 7's `spatial_cluster` and per-coordinate climate-normal
machinery documented in Section 8's Executive Summary above.

### Key findings

**1. The 35°C heat-threshold convention would have silently failed here — caught
before it shipped, not after.** `tmax_30d` never exceeds 34.97°C anywhere in this
dataset (train or test). A feature engineer following the traditional WHO-style
"days over 35°C" convention literally would have produced a second constant-zero
column, reproducing `hot_days_30d`'s exact problem under a different name. The
data-driven, train-only 90th-percentile threshold used instead (~30.7°C) avoids this
trap by construction, and its near-equal train/test flag rates (9.9% vs. 11.9%)
suggest it captures a real, generalizing distinction within this dataset's own range
rather than an arbitrary cutoff.

**2. The climate-anomaly baseline's 3-tier fallback is not a defensive
over-engineering — the sparse middle tier fires for real, verified rows.** A
meaningful share of `(spatial_cluster, month)` combinations seen in training have
fewer than 5 records, and some combinations never occur at all. Both training and
test rows genuinely land in the `group`-only fallback tier, confirmed by inspecting
`tavg_30d_anomaly_baseline_level`'s value counts directly rather than assuming the
finest-grained tier would always be available.

**3. Two honest non-features are recorded as deliberate scope decisions, not gaps.**
Neither a true daily heat-day count nor a consecutive dry-spell length is computable
from the provided window-level aggregates — both are documented in the provenance
table as `N/A — not implemented` with the specific reason, rather than shipped as
features that would look meaningful without actually measuring what their names
imply. This is consistent with this project's established stance (e.g. Stage 4's
rejection of a fabricated rainy-season window) that a documented gap is preferable to
a plausible-looking but hollow feature.

**4. Every learned-baseline feature (heat threshold, climate anomalies) follows the
same fit-on-train-only, lookup-at-transform pattern established by Stage 7's
`SpatialClusterFeaturizer`.** This is not a stylistic preference — it's the only
version of these features that is actually deployable: at real prediction time there
is no batch of other test records to average a baseline over for a single new record,
only whatever was learned in advance from training data. `compute_heat_threshold`
and `ClimateAnomalyFeaturizer.fit` are both called exactly once, on `train_full`
only, and their outputs are reused verbatim against test.

### Decisions recorded

- **Heat-exceedance threshold: 90th percentile of train `tmax_30d`, fit on train
  only, reused verbatim for test/production** — never the fixed 35°C convention for
  this dataset.
- **Climate anomaly baseline: `(spatial_cluster, month)` → `spatial_cluster` →
  global, min bucket size 5, fit-on-train-only** — the same leakage-safety pattern as
  Stage 7, applied to a second class of learned feature.
- **`hot_days_30d` dropped**, `rain_ratio_*`/`rain_intensity_30d` NaN (not 0/inf) on
  zero-denominator rows, both re-verified against the real data at run time rather
  than hardcoded from a historical finding.
- **No dry-spell or true daily-heat-count feature is shipped** — recorded as a
  documented data limitation in `docs/feature_provenance.md`, not a TODO.
- **`docs/feature_provenance.md` is the authoritative leakage-timing reference**
  going forward — Stage 9 and every later modeling stage should consult it before
  treating a derived feature as leakage-free by default.

### Implications for the rest of the project

- **Stage 9 (interactions & encoding)** now has two independently-verified
  leakage-safe feature families (Stage 7 spatial, Stage 8 climate) to combine —
  `spatial_cluster × zone`, anomaly-magnitude interactions, and out-of-fold target
  encoding of `spatial_cluster` are all well-defined next steps.
- **Phase 4 (baselines & model zoo)** inherits a feature set where every
  learned-baseline column is already leakage-safe by construction — no additional
  "did this leak test information" audit should be needed for `tmax_30d_exceeds_flag`,
  `tavg_30d_anomaly`, or `rain_sum_90d_anomaly` specifically, though the provenance
  table's `Medium`-risk NDVI entries (composite lag) are still worth a second look
  once Stage 19's deployment framing is built out.
- **A concrete open risk carried forward:** the anomaly baseline's `group`-tier
  fallback, while verified working, is a coarser signal than the seasonally-specific
  one — if Phase 4 finds `tavg_30d_anomaly`/`rain_sum_90d_anomaly` underperforming
  expectations, checking whether affected rows are disproportionately `group`-tier
  (rather than `group_month`-tier) is a natural first diagnostic, not a reason to
  abandon the feature.

### Review findings (post-build adversarial pass)

A dedicated review pass, run before this stage was considered final, found and
fixed four issues and flagged one design consideration for later:

1. **Self-inclusion bias in the anomaly baseline (fixed).** `fit_transform`
   originally computed each `(cluster, month)`/cluster bucket mean using every row
   in that bucket, including the row currently being transformed — a training row's
   own value contributed to its own baseline. This does not leak the target (it's a
   bias in a covariate statistic, not `is_climate_sensitive` itself), but it
   systematically shrinks the anomaly magnitude toward zero by a factor of
   `(n-1)/n` for a bucket of size `n` — verified as a full 33% shrinkage for
   `spatial_cluster` 3's 3 training records (in-sample anomalies `[0.296, 0.445,
   -0.741]` vs. the correct leave-one-out values `[0.444, 0.668, -1.111]`). Fixed:
   `fit_transform` now computes true leave-one-out baselines, falling back a tier
   when excluding a row would empty its bucket. `transform()` on genuinely new data
   (test, production) was already leakage-free and required no change — this is a
   deliberate, documented divergence from the sklearn convention that
   `fit_transform(X) == fit(X).transform(X)`, following the same precedent as
   `category_encoders`' `LeaveOneOutEncoder`.
2. **Bare-string `value_cols` footgun (fixed).** `ClimateAnomalyFeaturizer(value_cols="tavg_30d")`
   — an easy typo (forgetting to wrap a single column name in a list) — previously
   iterated the string character-by-character, silently trying to look up columns
   named `"t"`, `"a"`, `"v"`, etc. Now raises a clear `TypeError` immediately.
3. **Misleading docstring (fixed).** `add_rainfall_rate_features`'s docstring
   implied `rain_sum_7d`'s zero-rows were a "zero denominator" case like
   `rain_sum_30d`'s; `rain_sum_7d` is in fact never used as a denominator anywhere
   in this module (only as `rain_ratio_acute_medium`'s numerator, where zero is a
   perfectly well-defined ratio, not an undefined one). Corrected.
4. **Non-finite threshold guard (fixed).** `add_heat_exceedance_features` now
   rejects a NaN/inf `threshold` with a clear error instead of silently producing an
   all-zero flag column.
5. **`spatial_cluster`/anomaly redundancy (documented, not a bug).** `spatial_cluster`
   was itself built by clustering each location's average `tavg_30d` and
   `rain_sum_90d` (Stage 7), so a `spatial_cluster`-relative anomaly structurally
   absorbs some of the very signal the clustering was optimized to minimize. The
   feature still captures genuine record-level/seasonal deviation the location-level
   clustering never saw, so this isn't wrong — but it means the feature's eventual
   Tier 2 lift shouldn't be over-interpreted without comparing against a
   climate-independent `group_col` (e.g. `zone`) as a Phase 4 ablation.

All 59 climate tests (135 total across the project) pass after these fixes, including
new tests that hand-verify the leave-one-out arithmetic and the tier-fallback edge
case where excluding a bucket's lone member would divide by zero.

### Overall verdict

Stage 8 is **complete and validated**: all four tracks are implemented, tested against
both synthetic edge cases and the real data's actual sparsity/threshold quirks, every
learned-baseline feature follows the leakage-safe fit-on-train-only pattern, two real
data limitations are honestly documented rather than papered over, and the provenance
table is saved as the project's ongoing leakage-timing reference. Ready to proceed to
Stage 6 (temporal/demographic) and Stage 9 (interactions & encoding).